# RT-DETR Football Fine-Tuning — Google Colab

## 📤 Başlamadan Önce Drive'a Yüklenecekler

Sadece şunu yükle:
```
MyDrive/football_analysis/football-players-detection-1.zip
```
Unzip işlemi Cell 2'de otomatik yapılır.

> **Runtime:** Çalışma zamanı → Çalışma zamanı türünü değiştir → **A100 GPU**

## 1. Kurulum ve Drive Mount

In [ ]:
!pip install ultralytics -q

from google.colab import drive
drive.mount('/content/drive')
print('Drive bağlandı ✅')

## 2. ZIP'i Aç (Drive İçinde)

ZIP dosyan şurada olmalı:
`MyDrive/football_analysis/football-players-detection-1.zip`

In [ ]:
import zipfile, os

ZIP_PATH   = '/content/drive/MyDrive/football_analysis/football-players-detection-1.zip'
EXTRACT_TO = '/content/drive/MyDrive/football_analysis/training/'

os.makedirs(EXTRACT_TO, exist_ok=True)
DATASET_DIR = os.path.join(EXTRACT_TO, 'football-players-detection-1')

if os.path.exists(DATASET_DIR):
    print(f'Zaten açılmış, tekrar açılmadı → {DATASET_DIR} ✅')
else:
    print('ZIP açılıyor... (birkaç dakika sürebilir)')
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_TO)
    print(f'Unzip tamamlandı → {DATASET_DIR} ✅')

# Klasör içeriğini doğrula
for split in ('train', 'valid', 'test'):
    img_dir = os.path.join(DATASET_DIR, split, 'images')
    if os.path.exists(img_dir):
        print(f'  {split}/images → {len(os.listdir(img_dir))} dosya ✅')
    else:
        print(f'  {split}/images → BULUNAMADI ❌ (ZIP yapısını kontrol et!)')

## 3. GPU ve Path Kontrolü

In [ ]:
import torch, yaml

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {vram:.1f} GB')

DRIVE_ROOT  = '/content/drive/MyDrive/football_analysis'
MODELS_DIR  = os.path.join(DRIVE_ROOT, 'models')
RUNS_DIR    = os.path.join(DRIVE_ROOT, 'training', 'runs')

print(f'\nDataset : {DATASET_DIR}')
print(f'Models  : {MODELS_DIR}')

## 4. data_colab.yaml Oluştur

In [ ]:
# Orijinal data.yaml'daki ../relative path sorununu absolute ile çöz
colab_cfg = {
    'train': os.path.join(DATASET_DIR, 'train', 'images'),
    'val'  : os.path.join(DATASET_DIR, 'valid', 'images'),
    'test' : os.path.join(DATASET_DIR, 'test',  'images'),
    'nc'   : 4,
    'names': ['ball', 'goalkeeper', 'player', 'referee']
}

DATA_YAML = os.path.join(DATASET_DIR, 'data_colab.yaml')
with open(DATA_YAML, 'w') as f:
    yaml.dump(colab_cfg, f)

print(f'data_colab.yaml → {DATA_YAML} ✅')

## 5. Batch Size Seç

In [ ]:
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0

if vram_gb >= 35:
    BATCH = 16   # A100 (40GB)
elif vram_gb >= 14:
    BATCH = 8    # T4 (15GB) / V100 (16GB)
else:
    BATCH = 4

print(f'VRAM: {vram_gb:.1f} GB → batch={BATCH} seçildi')

## 6. Model Yükle

In [ ]:
from ultralytics import RTDETR
model = RTDETR('rtdetr-l.pt')  # Otomatik indirir
print(model.info())

## 7. Fine-Tune

> ⚠️ `augment=True` buraya yazılmaz — o sadece `model.predict()` için TTA parametresidir!

In [ ]:
results = model.train(
    data      = DATA_YAML,
    epochs    = 50,
    imgsz     = 1280,
    batch     = BATCH,
    optimizer = 'AdamW',
    lr0       = 1e-4,
    lrf       = 0.01,
    patience  = 15,
    mosaic    = 1.0,
    fliplr    = 0.5,
    flipud    = 0.0,
    degrees   = 0.0,
    project   = RUNS_DIR,
    name      = 'rtdetr_football',
    exist_ok  = True,
    device    = 0,
    verbose   = True,
)

## 8. Modeli Drive'a Kaydet ve İndir

In [ ]:
import shutil
from google.colab import files

os.makedirs(MODELS_DIR, exist_ok=True)

best_pt   = results.save_dir / 'weights' / 'best.pt'
output_pt = os.path.join(MODELS_DIR, 'rtdetr_best.pt')

shutil.copy(best_pt, output_pt)
print(f'Drive\'a kaydedildi → {output_pt} ✅')
print(f'Boyut: {os.path.getsize(output_pt)/1e6:.1f} MB')

# Direkt bilgisayara indirmek istersen:
# files.download(output_pt)

## 9. Validation Metrikleri

In [ ]:
val_results = model.val(data=DATA_YAML, imgsz=1280, verbose=True)

print('\n─── Validation Sonuçları ───')
print(f'mAP50    : {val_results.box.map50:.4f}')
print(f'mAP50-95 : {val_results.box.map:.4f}')

class_names = ['ball', 'goalkeeper', 'player', 'referee']
if hasattr(val_results.box, 'ap_class_index') and val_results.box.ap_class_index is not None:
    print('\nSınıf bazlı AP50:')
    for i, cls_idx in enumerate(val_results.box.ap_class_index):
        name = class_names[cls_idx] if cls_idx < len(class_names) else f'cls_{cls_idx}'
        print(f'  {name:<12}: {val_results.box.ap50[i]:.4f}')